# 🧠 Week 1 Exercise — AI Technical Tutor
**LLM Engineering Course — Ed Donner**  
*Author: Abdulrahman Alanani*

---

A technical question-answering tool that explains any programming or AI concept in depth.  
Powered by two backends running side by side:
- **OpenAI GPT** (via API) with real-time streaming
- **Ollama Gemma4** (local, free, no API key needed)

**LLM Engineering concepts covered:**
- System prompt engineering for structured, expert responses
- User prompt templating with f-strings
- OpenAI streaming API (`stream=True` + `update_display`)
- Local LLM inference via Ollama
- Jupyter Markdown rendering of LLM output

---

**How to use:** Edit the `question` variable in Cell 4, then run all cells top to bottom.


## Cell 1 — Install dependencies
Run once. Safe to re-run — will skip already-installed packages.


In [1]:
# Install required packages (run once)
!pip install openai python-dotenv ollama --quiet
print('✅ Dependencies ready')


✅ Dependencies ready


## Cell 2 — Imports & environment setup
Load the API key from `.env` and initialize the OpenAI client.

> 💡 `load_dotenv(override=True)` re-reads your `.env` file every time this cell runs —
> useful if you rotate your API key without restarting the kernel.


In [2]:
import os
import ollama
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Load API key from .env file
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    print('✅ API key looks good')
else:
    print('⚠️  API key missing or invalid — check your .env file')

openai = OpenAI()
print('✅ OpenAI client ready')


✅ API key looks good
✅ OpenAI client ready


## Cell 3 — Model constants
Centralizing model names as constants means you change them in one place only.
This is a basic but important production habit.


In [3]:
# Model identifiers — change here to switch models globally
MODEL_GPT    = 'gpt-5.4-mini'   # OpenAI API model (paid)
MODEL_OLLAMA = 'gemma4'          # Local Ollama model (free) — must be pulled first
                                  # To pull: ollama pull gemma4

print(f'GPT model    : {MODEL_GPT}')
print(f'Ollama model : {MODEL_OLLAMA}')


GPT model    : gpt-5.4-mini
Ollama model : gemma4


## Cell 4 — Your question ✏️
**Edit this cell's code to ask anything technical.**  
Run from this cell downward to get a fresh answer without re-running setup.


In [4]:
# ✏️ EDIT THIS — type your technical question here
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

print('✅ Question set')
print(f'   {len(question)} characters')


✅ Question set
   117 characters


## Cell 5 — System prompt
The system prompt defines the model's **identity, expertise, and response structure**.  
This is the most important engineering decision in any LLM app — a well-crafted system
prompt is the difference between a generic chatbot and a focused, expert tool.

> 💡 Notice how the system prompt specifies *both* what to cover (topics)
> *and* how to respond (structure, depth, style). Separating these two concerns
> from the user's question is the correct pattern.


In [5]:
system_prompt = """
You are an expert AI Technical Tutor and Software Engineering Educator.

Your role is to explain technical concepts clearly, accurately, and deeply across:
- Python programming
- Software engineering
- Data structures & algorithms
- APIs and backend systems
- Machine learning and deep learning
- LLM engineering
- RAG systems and AI agents
- Prompt engineering
- MLOps and deployment
- Data science

Your explanations must:
- Be technically precise
- Be beginner-friendly when needed
- Scale in depth depending on question complexity
- Focus on conceptual understanding, not only final answers
- Use practical engineering examples
- Explain WHY something works, not only WHAT it does
- Highlight tradeoffs, common mistakes, and best practices
- Use clean structure and formatting

When answering:
1. Start with the big-picture intuition
2. Break the concept into smaller parts
3. Explain important terminology
4. Show step-by-step reasoning when relevant
5. Include code examples when useful
6. Explain real-world engineering usage
7. Mention performance or scalability considerations if applicable
8. End with a concise summary

If the question involves code:
- Explain the code line-by-line when appropriate
- Identify bugs, inefficiencies, or bad practices
- Suggest cleaner or more production-ready alternatives
- Mention complexity and performance implications

If the question is ambiguous:
- State reasonable assumptions clearly
- Ask clarifying questions only when necessary

Do not give shallow generic explanations.
Optimize for deep learning and practical understanding.
"""

print('✅ System prompt ready')
print(f'   {len(system_prompt)} characters')


✅ System prompt ready
   1584 characters


## Cell 6 — User prompt
The user prompt wraps the question with a **response structure template**.  
Telling the model exactly what sections to include produces consistent, scannable output.

> 💡 The system prompt sets the agent's expertise. The user prompt sets the task format.
> Keeping these separate makes both easier to iterate on independently.


In [6]:
user_prompt = f"""
Please provide a detailed technical explanation for the following question.

Question:
{question}

Structure your response with:
- High-level intuition
- Core concepts
- Step-by-step explanation
- Practical examples
- Common mistakes or misconceptions
- Real-world applications
- Summary

Use clear and professional technical explanations.
"""

print('✅ User prompt ready')
print(f'   {len(user_prompt)} characters')


✅ User prompt ready
   448 characters


## Cell 7 — Build the messages list
The OpenAI API expects a list of messages with `role` and `content`.  
The same messages list is reused for both the OpenAI and Ollama calls — 
both APIs share this format, which is why Ollama chose to be OpenAI-compatible.


In [7]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": user_prompt},
]

print('✅ Messages list ready')
print(f'   Roles: {[m["role"] for m in messages]}')


✅ Messages list ready
   Roles: ['system', 'user']


## Cell 8 — GPT answer (with streaming) 🌊
Using `stream=True` means the response renders **word by word** as it arrives —
instead of waiting for the full response before displaying anything.

> 💡 **Why streaming matters in production:** Users perceive streamed responses as
> faster and more responsive, even if total generation time is identical.
> All major AI products (ChatGPT, Claude, Copilot) stream by default for this reason.

> 💡 `update_display` replaces the previous output in-place rather than appending
> new lines — this is the correct Jupyter pattern for live-updating content.


In [8]:
print(f'🤖 Asking {MODEL_GPT} (streaming)...\n')

stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True,     # enables token-by-token streaming
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    # Each chunk contains a delta — the next token(s) in the response
    response += chunk.choices[0].delta.content or ''

    # Strip markdown fences — Jupyter renders raw Markdown natively
    response = response.replace("```", "").replace("markdown", "")

    # Update the display in-place with the growing response
    update_display(Markdown(response), display_id=display_handle.display_id)


🤖 Asking gpt-5.4-mini (streaming)...



## High-level intuition

This code:

python
yield from {book.get("author") for book in books if book.get("author")}


takes a collection of `book` objects/dictionaries, extracts the `"author"` value from each one **if it exists and is truthy**, removes duplicates by using a **set**, and then **yields each unique author** from the current generator/function.

In plain English:

- look through all books
- keep only books that have a non-empty author
- collect the author names into a set
- yield each unique author one by one

---

## Core concepts

To understand this line, we need to break down three pieces:

### 1. Set comprehension

python
{book.get("author") for book in books if book.get("author")}


This creates a **set** of author names.

A set:
- automatically removes duplicates
- is unordered
- only stores unique elements

So if multiple books have `"author": "Alice"`, the set will contain `"Alice"` only once.

---

### 2. `book.get("author")`

`book` is likely a dictionary.

python
book.get("author")


means:
- return the value associated with `"author"`
- if `"author"` is missing, return `None` instead of raising a `KeyError`

This is safer than:

python
book["author"]


because the latter would crash if the key is missing.

---

### 3. `yield from`

`yield from iterable` delegates yielding to another iterable.

Instead of writing:

python
for author in some_iterable:
    yield author


you can write:

python
yield from some_iterable


It is a concise way to yield every item from another iterable.

---

## Step-by-step explanation

Let’s rewrite the expression in a more expanded form.

### Original

python
yield from {book.get("author") for book in books if book.get("author")}


### Step 1: Iterate through `books`

Assume `books` is a list of dictionaries like:

python
books = [
    {"title": "Book A", "author": "Alice"},
    {"title": "Book B", "author": "Bob"},
    {"title": "Book C", "author": "Alice"},
    {"title": "Book D"},
    {"title": "Book E", "author": ""}
]


### Step 2: Filter by truthy author

The condition:

python
if book.get("author")


keeps only books where the author value is truthy.

That means it excludes:
- `None`
- empty string `""`
- `False`
- `0`
- empty containers like `[]`, `{}`, `set()`

So in the example above:
- `"Alice"` → kept
- `"Bob"` → kept
- `"Alice"` again → kept for now
- missing author → excluded
- empty author string → excluded

### Step 3: Build a set of authors

The set comprehension results in:

python
{"Alice", "Bob"}


Duplicates are removed automatically.

### Step 4: `yield from` each author

Then:

python
yield from {"Alice", "Bob"}


yields each element from the set.

The generator/function emits:
- `"Alice"`
- `"Bob"`

The order is **not guaranteed**, because sets are unordered collections.

---

## Practical examples

### Example 1: Basic usage

python
def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}


Usage:

python
books = [
    {"title": "Clean Code", "author": "Robert C. Martin"},
    {"title": "The Pragmatic Programmer", "author": "Andrew Hunt"},
    {"title": "Another Book", "author": "Robert C. Martin"},
    {"title": "Untitled"},
]

print(list(unique_authors(books)))


Possible output:

python
['Robert C. Martin', 'Andrew Hunt']


Order may vary.

---

### Example 2: Equivalent expanded version

The code is roughly equivalent to:

python
def unique_authors(books):
    authors = set()
    for book in books:
        author = book.get("author")
        if author:
            authors.add(author)
    for author in authors:
        yield author


This version is more verbose, but easier to understand for many readers.

---

### Example 3: If you want stable order

Because sets do not preserve order, this code is not suitable if you need authors in the order they first appear.

For stable order, use a different approach:

python
def unique_authors_in_order(books):
    seen = set()
    for book in books:
        author = book.get("author")
        if author and author not in seen:
            seen.add(author)
            yield author


This preserves first-seen order.

---

## Common mistakes or misconceptions

### 1. Assuming the output order is predictable

A major misconception is that the yielded authors will appear in input order.

They will not, because:
- the intermediate collection is a **set**
- sets are unordered

If order matters, this code is the wrong choice.

---

### 2. Confusing `yield from` with `return`

`yield from` does **not** return the set.

It yields items from it one by one.

So this is a generator-style behavior, not a normal function return.

---

### 3. Thinking it includes all authors, even empty ones

The condition:

python
if book.get("author")


filters out falsy authors.

So this excludes:
- `None`
- `""`

If you want to include empty strings but exclude only missing keys, you would need a different condition.

For example:

python
if "author" in book


---

### 4. Calling `book.get("author")` twice

The expression evaluates `book.get("author")` twice:
- once in the `if` condition
- once in the set element expression

This is fine for small code, but it is slightly inefficient and can be cleaner if stored once:

python
def unique_authors(books):
    for book in books:
        author = book.get("author")
        if author:
            yield author


If uniqueness is required, you’d still need a `set` to track seen authors.

---

### 5. Assuming `yield from` is necessary here

`yield from` is concise, but not always needed. It is mainly useful when delegating to another iterable or generator.

In this case, it is a compact way to yield from the set, but a plain `for` loop would be just as valid.

---

## Real-world applications

This pattern is useful when:

### 1. Deduplicating values from records
For example:
- authors from books
- tags from articles
- categories from products
- users from logs

---

### 2. Streaming values from a generator
If the surrounding function is itself a generator, `yield from` keeps the code concise and readable.

---

### 3. Data preprocessing
In ETL or data pipelines, you may want to extract unique non-empty fields from a list of dictionaries before storing or analyzing them.

---

### 4. API response normalization
If an API returns nested records, this pattern can extract a unique set of valid attribute values.

---

## Important tradeoffs

### Pros
- Very concise
- Removes duplicates automatically
- Skips empty/missing authors
- Works naturally in generator contexts

### Cons
- Unordered output
- Slightly harder to read for beginners
- Builds the full set in memory before yielding anything
- Calls `book.get("author")` twice per book

If the data is large or order matters, a more explicit loop is usually better.

---

## Summary

The code:

python
yield from {book.get("author") for book in books if book.get("author")}


does the following:

1. Iterates over `books`
2. Extracts each book’s `"author"`
3. Keeps only truthy author values
4. Stores them in a set to remove duplicates
5. Yields each unique author from the generator

### Key takeaway:
It produces **unique, non-empty authors**, but the order is **not guaranteed** because it uses a set.

If you want, I can also show:
- an equivalent version written in plain loops
- a version that preserves input order
- a line-by-line AST-level breakdown of how Python parses this expression

## Cell 9 — Ollama answer (local, free) 🖥️
Runs the same question through a **local model** using Ollama —
no API key, no cost, no data leaving your machine.

> 💡 **Prerequisite:** Ollama must be installed and the model pulled before running.
> ```bash
> # Install: https://ollama.com
> ollama pull gemma4
> ```

> 💡 **Engineering insight:** Comparing GPT and a local model on the same prompt is
> a practical way to evaluate quality vs. cost tradeoffs — a skill you'll use constantly
> when building real AI products.


In [9]:
print(f'🖥️  Asking {MODEL_OLLAMA} via Ollama (local)...\n')

response_ollama = ollama.chat(
    model='gemma4',
    messages=messages,   # same messages list — Ollama uses the same format
)

reply = response_ollama['message']['content']
display(Markdown(reply))


🖥️  Asking gemma4 via Ollama (local)...



This is an excellent, highly dense piece of code that touches upon several advanced Python concepts: **Generators, Comprehensions (Sets and Generators), and Yield Statements.**

As an expert technical tutor, I will explain not only what the code does but also *why* it performs that way, addressing the subtle performance implications and common best practices.

***

### High-Level Intuition (The Goal)

The entire line of code is designed to efficiently process a list of complex data records (books, which are dictionaries) and extract only the unique author names, yielding them one by one as they are found.

**The core idea is:** "Go through every book. If the book has an author, grab that author's name, and make sure we only yield that name once, without loading all names into memory simultaneously."

### Core Concepts Explained

To fully understand this line, we must understand four key concepts:

#### 1. Generators and `yield from`
*   **Generator:** A generator is a specialized iterable that produces a sequence of results over time, rather than calculating and storing the entire sequence in memory at once. Generators use the `yield` keyword.
*   **`yield from`:** This is a statement introduced in Python 3.3. It is syntactic sugar that delegates to another generator or iterable. Essentially, it means: "Run this iterable (the thing on the right) and yield every item it produces, as if I had written a `for item in iterable: yield item` loop."
*   **Why it matters:** Generators are crucial for memory efficiency (scalability) when processing massive datasets, as they operate lazily (only calculating what is needed when asked).

#### 2. Comprehensions (Set vs. Generator)
*   **General Comprehension:** Python allows concise ways to create lists, sets, or dictionaries.
*   **Set Comprehension (`{...}`):** When you use curly braces `{...}`, you are creating a **Python `set`** object. A set guarantees that all elements are unique. This is the mechanism that ensures we only count each author once.
*   **Generator Comprehension (`(...)`):** When you use parentheses `(...)`, you are creating a **generator object**. A generator is memory-efficient because it does not build the entire sequence; it only stores the instructions on *how* to compute the next value.

#### 3. Dictionary Methods (`.get()`)
*   `book.get("author")` is the preferred way to access a dictionary key.
*   **Why use `.get()` instead of `['author']`?** If the key `"author"` does not exist in the `book` dictionary, using `['author']` will throw a `KeyError` and crash the program. Using `.get("author")` will gracefully return `None` (or a specified default value) if the key is missing, allowing the program to continue running.

#### 4. Truthiness Filtering (`if book.get("author")`)
*   In Python, `None`, `0`, empty strings (`""`), empty lists, etc., are considered "falsy" (they evaluate to `False` in a boolean context).
*   The filter `if book.get("author")` ensures that we only process books where the author field exists **and** where the author name is a non-empty, non-`None` value.

### Step-by-Step Explanation

Let's analyze the source of the `yield from` statement:
$$\text{Source} = \text{`\{book.get("author") for book in books if book.get("author")\}`}$$

**Step 1: Initialization and Iteration**
The code begins iterating through the `books` iterable (which we assume is a list of dictionaries). For each `book` dictionary, the code attempts to execute the rest of the comprehension.

**Step 2: The Filter (The `if` Clause)**
For the current `book`, it first evaluates the filter: `if book.get("author")`.
*   If `book.get("author")` returns `None` (key missing) or `""` (empty string), the `if` condition fails, and the current book is skipped entirely.
*   If it returns a string like `"J.K. Rowling"`, the condition succeeds, and the process continues.

**Step 3: The Expression (The Value to be Computed)**
If the filter passes, the expression `book.get("author")` is executed. This extracts the author's name.

**Step 4: The Container (The `{...}` Set Comprehension)**
The results of Step 3 are collected into a **set** due to the surrounding curly braces. The nature of a set ensures that if the same author name appears multiple times (e.g., two different books share the same author), the set will automatically discard the duplicates, storing only one instance of that author's name.

**Step 5: The Execution (The `yield from`)**
Finally, the outer function encounters `yield from {set_object}`.
1.  Python first creates the full, internal `set` object containing all the unique authors found in the `books` list.
2.  It then uses `yield from` to iterate over this set object and yields each unique author name one by one from the calling context.

---
### 🧠 Critical Technical Insight: The Set vs. Generator Trap (Optimization)

While the code as written *works*, it is not optimal for large-scale engineering applications because of the `set` comprehension.

**What the current code does:**
1.  It processes *all* $N$ books.
2.  It constructs the entire `set` object in memory.
3.  Then, it yields the results.

**The inefficiency:** Creating a set requires that all the data be processed and stored in memory *before* the first value can be yielded. If `books` is gigabytes large, this will consume significant memory unnecessarily.

**The Recommended Best Practice (The Fix):**
If the goal is lazy evaluation (processing data as it arrives, without storing all unique items simultaneously), you should use a **generator comprehension** and then wrap that in a function that handles the de-duplication logic, or simply use a standard `set` *outside* the generator structure if the inputs are small.

However, since the requirement is to *yield* (implying streaming and memory efficiency), a pure generator approach is ideal.

**If de-duplication is mandatory, but memory is a concern, you must use a separate tracking set:**

```python
# The optimized, memory-efficient approach
def get_unique_authors_stream(books):
    seen_authors = set()
    for book in books:
        author = book.get("author")
        if author and author not in seen_authors:
            seen_authors.add(author)
            yield author # Yield immediately when a new author is found
```
This alternative achieves the same result (unique authors yielded one by one) but maintains **$O(1)$** amortized space complexity per yield, rather than potentially $O(A)$ space complexity (where $A$ is the number of unique authors) to build the intermediate set.

### Practical Example

Assume the following input data:

```python
books = [
    {"title": "Book A", "author": "Alice"},  # Valid
    {"title": "Book B", "author": "Bob"},    # Valid
    {"title": "Book C", "author": "Alice"},  # Duplicate Author
    {"title": "Book D", "author": None},    # Filtered out (None)
    {"title": "Book E"},                    # Filtered out (Missing key)
    {"title": "Book F", "author": ""},      # Filtered out (Empty string/Falsy)
]
```

**Execution Trace:**

1.  The comprehension iterates through `books`.
2.  **Book A:** `"Alice"` is extracted. The set becomes `{"Alice"}`.
3.  **Book B:** `"Bob"` is extracted. The set becomes `{"Alice", "Bob"}`.
4.  **Book C:** `"Alice"` is extracted. The set remains `{"Alice", "Bob"}` (no change, due to uniqueness).
5.  **Book D:** Filter fails (`None` is falsy). Skipped.
6.  **Book E:** Filter fails (`.get()` returns `None`, which is falsy). Skipped.
7.  **Book F:** Filter fails (`""` is falsy). Skipped.
8.  The resulting `set` is `{"Alice", "Bob"}`.
9.  `yield from` executes, yielding the elements of the set.

**Output (The values yielded):**
```
"Alice"
"Bob" 
# (Order may vary since sets are inherently unordered)
```

### Common Mistakes and Misconceptions

1.  **Confusing Set Comprehensions vs. Generator Comprehensions:**
    *   **Mistake:** Using `{...}` when you only need lazy evaluation.
    *   **Impact:** Inefficient memory usage for streaming data.
    *   **Fix:** Use `(...)` if you want the structure to be consumed one item at a time, *unless* uniqueness is the primary requirement, in which case you must use the explicit loop with a tracking set (as shown in the optimization section) if memory is critical.

2.  **Ignoring the Filter:**
    *   **Mistake:** Omitting `if book.get("author")`.
    *   **Impact:** The comprehension will attempt to process books where the author key is missing, which can sometimes lead to logical errors or unexpected behavior if the author value is not strictly a string.

3.  **Assuming Order:**
    *   **Mistake:** Assuming that the items yielded will appear in the original list order.
    *   **Impact:** Because the underlying structure is a `set`, the iteration order of the yielded items is non-deterministic (unpredictable).

### Real-World Engineering Usage

This pattern (unique filtering over a stream) is highly common in backend systems and data pipelines:

1.  **API Rate Limiting:** If you are processing user request logs, you might use this pattern to uniquely identify all user IDs that hit the API endpoint within a short window, ensuring you don't track the same user multiple times for rate limit calculations.
2.  **Data Validation Pipeline:** When processing a batch of submitted records (e.g., user profiles), you use this logic to collect a unique list of all phone numbers or email addresses that were submitted, allowing you to send a single, consolidated notification to the system admin.
3.  **Database Indexing:** If you need to create a unique index table of a field (like `author`) from a list of database records, this comprehension pattern simulates that grouping and uniqueness finding efficiently in Python memory.

### Summary

| Component | Type | Purpose | Efficiency Note |
| :--- | :--- | :--- | :--- |
| **Outer Function** | Generator | Yields items one by one, maintaining memory efficiency. | Excellent for large datasets. |
| **`yield from`** | Statement | Delegates iteration to the result of the comprehension. | Clean syntax for generator chaining. |
| **`{...}`** | Set Comprehension | Collects results and enforces **uniqueness**. | **Memory Cost:** Builds the entire set in memory first. |
| **`book.get("author")`** | Dictionary Method | Safely retrieves the author name; returns `None` if missing. | Prevents `KeyError` crashes. |
| **`if ...`** | Filter | Filters out entries where the author name is missing (`None`) or empty (`""`). | Ensures data quality. |